# Instrument subsystems

The first six notebooks stayed on the single-arm imaging path: pupil, mask,
focal plane. A real instrument branches. This notebook is a short how-to for the
three subsystems that hang off the propagation core, each shown as a minimal
worked example:

- **Splitting the beam** into a science arm and a wavefront-sensing arm with a
  {class}`~physicaloptix.BeamSplitter` inside an {class}`~physicaloptix.OpticalSystem`.
- **Reading out a detector** -- turning a noiseless intensity into noisy
  electron counts with {func}`~physicaloptix.read_detector`.
- **Dispersing with an integral-field spectrograph** -- a wave-optics
  {class}`~physicaloptix.LensletChain` and the PSFlet template pack it emits.

These are entry points, not deep dives; each links onward to where its full
story lives.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import json

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

import physicaloptix as po
from physicaloptix import (
    BeamSplitter,
    Branch,
    Field,
    Fraunhofer,
    Grid,
    LensletChain,
    OpticalPath,
    OpticalSystem,
    PlaneKind,
    SampledOptic,
    Stage,
    psflet_pack,
    read_detector,
)
from physicaloptix.ifs import psflet_template

print("physicaloptix", po.__version__)

NPUP, NFOC, PSCALE = 64, 128, 0.25
pupil = Grid.pupil(NPUP)
focal = Grid.focal(NFOC, PSCALE)
x = np.asarray(pupil.coords)
xg, yg = np.meshgrid(x, x)
aperture = ((xg**2 + yg**2) <= 0.25).astype(complex)
flat = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.PUPIL)
ext = [-NFOC / 2 * PSCALE, NFOC / 2 * PSCALE] * 2

## Splitting the beam

A coronagraph instrument sends most of its light to the science camera and taps
a fraction for a wavefront sensor. A {class}`~physicaloptix.BeamSplitter` models
that split at a plane, conserving energy: the lossless convention puts the two
ports in quadrature,

$$ t = \sqrt{1 - R}, \qquad r = \mathrm{i}\,\sqrt{R}, $$

so a fraction $R$ of the intensity goes to the reflect port and $1 - R$ to the
transmit port. `BeamSplitter.energy(R, ...)` builds a grey (achromatic) split;
`BeamSplitter.dichroic(...)` builds a wavelength-dependent one for a color
split.

An {class}`~physicaloptix.OpticalSystem` wires a shared trunk to named branches
through the splitter, propagates the trunk once, and runs each branch on its
port. Here the trunk is a telescope stop, 90 percent of the light forms the
science image, and 10 percent forms a wavefront-sensor image.

In [ ]:
stop = SampledOptic(
    transmission=jnp.asarray((xg**2 + yg**2 <= 0.45**2).astype(float)),
    grid=pupil,
    plane=PlaneKind.PUPIL,
)
trunk = OpticalPath(stages=(Stage("stop", stop),))
split = BeamSplitter.energy(0.1, grid=pupil, plane=PlaneKind.PUPIL)  # 10% reflected

science = OpticalPath(stages=(Stage("img", Fraunhofer(grid_in=pupil, grid_out=focal)),))
wfs = OpticalPath(stages=(Stage("img", Fraunhofer(grid_in=pupil, grid_out=focal)),))

system = OpticalSystem(
    trunk=trunk,
    split=split,
    branches=(Branch("science", "transmit", science), Branch("wfs", "reflect", wfs)),
)
outs, _ = system.propagate(flat)

e_sci = float(jnp.sum(jnp.abs(outs["science"].data) ** 2))
e_wfs = float(jnp.sum(jnp.abs(outs["wfs"].data) ** 2))
print(f"science / wfs energy = {e_sci / e_wfs:.2f}  (90:10 split -> 9.0)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, name in zip(axes, ("science", "wfs")):
    img = np.abs(np.asarray(outs[name].data)) ** 2
    im = ax.imshow(
        img / img.max(),
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-6, 1),
        extent=ext,
        interpolation="none",
    )
    ax.set_title(f"{name} arm")
    ax.set_xlabel(r"$\lambda/D$")
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="normalized")
plt.show()

The two arms carry the same image morphology but a 9:1 energy ratio, and the
split conserves energy pointwise -- `system.propagate` runs the shared trunk
exactly once and checks the balance at the split. Real branches diverge after
the port: the science arm continues to a coronagraph, the sensing arm to a
Zernike or Shack-Hartmann sensor.

## Reading out a detector

Everything so far has been a noiseless intensity. A detector turns that into
integer-like electron counts with noise. {func}`~physicaloptix.read_detector`
applies the standard chain: the mean signal is

$$ \mu = \mathrm{QE}\cdot \Phi \cdot t_\mathrm{exp}\cdot I + d\,t_\mathrm{exp}, $$

for a normalized intensity $I$ (stellar peak scaled to one), a peak photon rate
$\Phi$, exposure time $t_\mathrm{exp}$, quantum efficiency QE, and dark rate
$d$; then Poisson shot noise on that mean and zero-mean Gaussian read noise on
top. We read the telescope PSF at a short and a long exposure to watch the
signal climb out of the noise.

In [ ]:
telescope = OpticalPath(
    stages=(Stage("img", Fraunhofer(grid_in=pupil, grid_out=focal)),)
)
airy, _ = telescope.propagate(flat)
psf = np.asarray(jnp.abs(airy.data) ** 2)
psf = psf / psf.max()  # normalized intensity, stellar peak = 1

key = jax.random.PRNGKey(0)
reads = {}
for t_exp in (1.0, 30.0):
    reads[t_exp] = np.asarray(
        read_detector(
            jnp.asarray(psf),
            key,
            flux=2.0e3,  # peak photon rate [photons / s]
            exposure_time=t_exp,
            read_noise_e=5.0,
            dark_e_per_s=0.02,
            quantum_efficiency=0.9,
        )
    )

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
im0 = axes[0].imshow(
    psf,
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-6, 1),
    extent=ext,
    interpolation="none",
)
axes[0].set_title("noiseless intensity")
fig.colorbar(im0, ax=axes[0], fraction=0.046, label="normalized")
for ax, t_exp in zip(axes[1:], (1.0, 30.0)):
    im = ax.imshow(
        reads[t_exp], origin="lower", cmap="inferno", extent=ext, interpolation="none"
    )
    ax.set_title(f"{t_exp:.0f} s read [electrons]")
    fig.colorbar(im, ax=ax, fraction=0.046, label=r"$e^-$")
for ax in axes:
    ax.set_xlabel(r"$\lambda/D$")
plt.tight_layout()
plt.show()

At 1 second the read-noise floor buries the diffraction rings and a few pixels
even read negative (Gaussian read noise on a near-zero mean); at 30 seconds the
rings emerge as the shot-noise-limited signal grows faster than the fixed read
noise. The function is a pure JAX transform of the intensity and the random key,
so an exposure-time or yield loop can `vmap` it over pointings and take
gradients through the mean signal. Full exposure-time and yield modeling lives
in [jaxedith](https://github.com/CoreySpohn/jaxedith).

## Dispersing with an integral-field spectrograph

An integral-field spectrograph (IFS) puts a lenslet array at the focal plane;
each lenslet samples one spatial point and a downstream disperser spreads its
light into a little spectrum -- a PSFlet -- on the detector.
{class}`~physicaloptix.LensletChain` is the wave-optics model of a single
lenslet, from the telescope exit pupil through the micro-pupil and spectrograph
stop to the detector. It is built from the pupil field and the lenslet geometry:
the pitch in $\lambda/D$ at a reference wavelength, and the micro-pupil
sampling.

We build one chain and evaluate its PSFlet template at three wavelengths.

In [ ]:
c = pupil.coords
disk = jnp.asarray((c[:, None] ** 2 + c[None, :] ** 2 <= 0.25).astype(float))
chain = LensletChain(
    disk,
    pupil,
    pitch_lod_ref=0.5,  # lenslet pitch [lambda/D] at the reference wavelength
    lam_ref_nm=1000.0,
    micropupil_px=1.5,  # detector pixels across the micro-pupil
    n_tile=48,
    illumination="psf",
)

offsets = jnp.arange(-6.0, 6.001, 0.25)  # detector sample offsets [px]
wavelengths = [700.0, 1000.0, 1300.0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, wl in zip(axes, wavelengths):
    plane, _ = psflet_template(chain, wl, offsets, n_quad=4)
    plane = np.asarray(plane)
    px = float(chain.px_per_diffraction(wl))
    im = ax.imshow(
        plane / plane.max(),
        origin="lower",
        cmap="magma",
        norm=LogNorm(1e-3, 1),
        interpolation="none",
    )
    ax.set_title(f"{wl:.0f} nm\n{px:.2f} px per diffraction unit")
    ax.set_xticks([]), ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="normalized")
plt.show()

The same lenslet makes a larger PSFlet at longer wavelength: the diffraction
spot scales with wavelength while the detector pitch is fixed, so `1000 nm`
spans more pixels than `700 nm`. That wavelength-to-position mapping across the
detector is what an IFS turns into a spectrum per spatial sample.

To hand these templates to a spectral extraction pipeline, `psflet_pack`
tabulates the chain over a wavelength list into a template pack, and records
how much of each lenslet's energy the micro-pupil and stop capture.

In [ ]:
pack = psflet_pack(chain, wavelengths, half_extent=6.0, step=0.25, n_quad=4)
meta = json.loads(pack["meta_json"])

print(f"templates : {pack['templates'].shape}  (field, wavelength, y, x)")
print(f"wavelengths: {list(np.asarray(pack['wavelengths_nm']))}")
print("capture efficiency (fraction of tile energy through the stop):")
for c_wl in meta["capture"]:
    print(f"  {c_wl['wavelength_nm']:.0f} nm : {c_wl['stop_capture']:.3f}")

The pack is a self-describing payload (`save_psflet_pack` writes it to an npz);
its `meta_json` records the exact chain geometry that generated it, so the
templates are reproducible. Building a full detector scene from these PSFlets and
extracting spectra from it is the job of the IFS pipeline that consumes the pack,
not of physicaloptix.

## Where these fit

The [architecture page](../explanation/architecture) places the branch model in
the wider design. Downstream, [coronagraphoto](https://coronagraphoto.readthedocs.io/)
assembles scenes onto a detector and [jaxedith](https://github.com/CoreySpohn/jaxedith)
turns reads into exposure-time and yield estimates. physicaloptix supplies the
diffraction each of them stands on.